# Week 1 · Day 1 — SELECT, WHERE, ORDER BY
**Goal:** Query a real table, filter it with purpose, sort it for insight

**Schedule:**
- Block 1 · 9:00–10:30am · Concept + Drills
- Block 2 · 11:00am–12:00pm · Apply + AI Audit

---
## Business First Prompt

> *Your manager says: "Sales feel slow lately but I don't know where the problem is. Figure out what's going on."*

Write 3-5 sentences below before touching any data:
- What would you measure first?
- Which table would you start with?
- What would "slow" actually look like in the data?


**Your answer:**

> I would compare current sales against the previous 3 months, looking at both order count and revenue since "slow" could mean fewer transactions, smaller order values, or both. I'd start with the orders table because it holds the core transaction data and can be grouped by date to reveal the trend over time. If sales are genuinely slowing, I'd expect to see a consistent decline in either total orders or total revenue across recent periods. Depending on the manager's context, I'd scope the window to the last 1–3 months, and ideally layer in a year-over-year comparison to control for seasonality. 

---
## Database Setup

Run this cell once at the start of every session this week. It connects to your weekly database.

**Your three tables this week:**
- `orders` — order_id, customer_id, order_date, status, total_amount, region
- `customers` — customer_id, name, email, signup_date, segment, country
- `order_items` — item_id, order_id, product_name, category, quantity, unit_price

Today you only use `orders`. All three tables come together on Wednesday with JOINs.

In [2]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('week1_ecommerce.db')
print('Connected to week1_ecommerce.db')

Connected to week1_ecommerce.db


---
## Concept — SELECT

Tells SQL which columns to return. `*` means all columns.

```sql
SELECT order_id, order_date, total_amount, status
FROM orders;

-- grab everything
SELECT *
FROM orders;
```

---
## Concept — WHERE

Filters rows. Only returns rows where the condition is true.
- Strings go in **single quotes**
- Numbers do **not** need quotes
- Use `AND` / `OR` for multiple conditions

```sql
WHERE status = 'completed'

WHERE status = 'completed' AND total_amount > 100

WHERE order_date >= '2024-01-01'
```

---
## Concept — ORDER BY

Sorts your results.
- `DESC` = largest to smallest, most recent first
- `ASC` = smallest to largest (default, rarely needed to write explicitly)

```sql
ORDER BY total_amount DESC

ORDER BY order_date DESC
```

---
## Concept — LIMIT

Caps the number of rows returned. Always use when exploring a new table.

```sql
SELECT *
FROM orders
ORDER BY order_date DESC
LIMIT 10;
```

---
## Drill 1 — First look at the table

Show the **10 most recent orders**.
Columns: order_id, order_date, status, total_amount.

Clauses you need: `SELECT` · `FROM` · `ORDER BY` · `LIMIT`

In [16]:
df = pd.read_sql_query("""

SELECT order_id, order_date, region, total_amount
FROM orders
ORDER BY order_date DESC 
LIMIT 10

""", conn)
display(df.head())

,order_id,order_date,region,total_amount
0,1030,2024-12-28,West,640.0
1,1029,2024-12-18,North,290.0
2,1028,2024-12-10,East,125.0
3,1027,2024-12-02,West,980.0
4,1026,2024-11-25,South,415.0


---
## Drill 2 — Filter by status

Show all **cancelled** orders.
Columns: order_id, order_date, total_amount, region. Most recent first.

Clauses you need: `SELECT` · `FROM` · `WHERE` · `ORDER BY`

In [ ]:
df = pd.read_sql_query("""

SELECT order_id, order_date, total_amount, region
FROM orders
WHERE status = 'cancelled'
ORDER BY order_date DESC
""", conn)
df

,order_id,order_date,total_amount,region
0,1028,2024-12-10,125.00,East
1,1022,2024-10-15,78.00,East
2,1017,2024-08-19,310.00,West
3,1011,2024-05-20,95.00,West
4,1006,2024-03-01,55.00,West
5,1002,2024-01-12,89.99,East


---
## Drill 3 — High-value completed orders

Find all **completed** orders where total_amount is **over $200**.
Columns: order_id, order_date, total_amount, region. Highest amount first.

Clauses you need: `SELECT` · `FROM` · `WHERE` with `AND` · `ORDER BY`

In [4]:
df = pd.read_sql_query("""

SELECT order_id, order_date, total_amount, region
FROM orders
WHERE total_amount > 200 AND status = 'completed'
ORDER BY total_amount DESC
""", conn)
df

,order_id,order_date,total_amount,region
0,1027,2024-12-02,980.0,West
1,1018,2024-09-02,890.0,East
2,1023,2024-10-22,720.0,West
3,1030,2024-12-28,640.0,West
4,1007,2024-03-15,620.0,South
5,1021,2024-10-07,560.0,South
6,1010,2024-05-05,510.0,North
7,1013,2024-06-25,480.0,West
8,1003,2024-01-20,430.5,West
9,1026,2024-11-25,415.0,South


---
## Drill 4 — One region, one time period

All **West** region orders placed **after Jan 1 2024**.
Columns: order_id, order_date, status, total_amount. Most recent first.

Clauses you need: `SELECT` · `FROM` · `WHERE` with `AND` · `ORDER BY`

Note: date comparisons use `>=` with the date in single quotes — `'2024-01-01'`

In [7]:
df = pd.read_sql_query("""

SELECT order_id, order_date, region, total_amount
FROM orders
WHERE region = 'West' AND order_date >= '2024-01-01'
ORDER BY order_date DESC

""", conn)
df

,order_id,order_date,region,total_amount
0,1030,2024-12-28,West,640.0
1,1027,2024-12-02,West,980.0
2,1023,2024-10-22,West,720.0
3,1020,2024-09-28,West,440.0
4,1017,2024-08-19,West,310.0
5,1013,2024-06-25,West,480.0
6,1011,2024-05-20,West,95.0
7,1008,2024-04-02,West,190.0
8,1006,2024-03-01,West,55.0
9,1003,2024-01-20,West,430.5


---
## Block 2 · 11:00am — Applied Challenge

> *Your manager says: "We think the West region is underperforming. Can you pull something quick that shows me what's going on with high-value orders out there?"*

Before writing any SQL, complete the sentence in the markdown cell below.

**I'm going to show ___ because ___:**

> I am going to show all high value orders from West region to identify metrics that could point towards underperformance like large amount of unfufilled orders or low volume of orders. However to evaluate if West is indeed underperforming company wide, I must first compare it to the other regions.

In [ ]:
# Your query — no single right answer
# The goal is a deliberate choice, not a perfect query

df = pd.read_sql_query("""
SELECT DISTINCT o.order_id,
         o.order_date, 
        o.total_amount,
        o.status, 
        oi.category
FROM orders o JOIN order_items oi ON o.order_id = oi.order_id
WHERE total_amount > 200 AND region = 'West'
ORDER BY order_date DESC
""", conn)
df

,order_id,order_date,total_amount,status,category
0,1030,2024-12-28,640.0,completed,Furniture
1,1030,2024-12-28,640.0,completed,Electronics
2,1027,2024-12-02,980.0,completed,Electronics
3,1027,2024-12-02,980.0,completed,Accessories
4,1023,2024-10-22,720.0,completed,Electronics
5,1020,2024-09-28,440.0,refunded,Furniture
6,1017,2024-08-19,310.0,cancelled,Furniture
7,1013,2024-06-25,480.0,completed,Accessories
8,1013,2024-06-25,480.0,completed,Electronics
9,1003,2024-01-20,430.5,completed,Electronics


**What did the data say?**
Would you tell your manager West has a problem based on this query alone? Why or why not?

From this query alone I would not be able to tell my manager that West has a uner performance problem as there are no clear identifiers of the claim. I would need to compare West's output to the other regions to understand what 'Underperformace' means for the company.

---
## Day 1 Checkpoint

Answer all three before opening Day 2. Plain English only — no SQL needed.

**1. What is the difference between WHERE and ORDER BY — what does each one actually do?**

> WHERE filteres the information needed from the intial table and ORDER BY organizes by value of choice in DESC or ASC order. 

**2. Your manager asks: "Show me all refunded orders over $500 from Q1 2024." Which clauses do you need and in what order?**

> I would use FROM first to call the table that holds the information i need then a WHERE clause to filter "refunded' orders over $500 in (Q1 2024) and finally a SELECT (*) to show the filtered output. 